In [1]:
import pandas as pd
import gzip
import json
from pm4py.utils import format_dataframe
from pm4py import write_xes
from tqdm.notebook import tqdm
from copy import deepcopy

In [32]:
dataset_json_path = r"D:\LTNcoder\.out\eventlogs\xbpic12-0.3-1.json.gz"
new_dataset_json_path = r"D:\LTNcoder\.out\eventlogs\bpic12-0.3-1.json.gz"
with gzip.open(dataset_json_path, "r") as f:
        data = f.read()
        j = json.loads(data.decode('utf-8'))
j_sampled = deepcopy(j)


In [33]:
print(j_sampled["cases"][0]["attributes"])
"""
{'AMOUNT_REQ': '20000',
'REG_DATE': '2011-10-01T00:38:44.546+02:00',
'concept:name': '173688',
'label': 'normal'}
"""
# check how many of the cases are normal and how many are anomlay
normal = 0
anomaly = 0
for case in j_sampled["cases"]:
    if case["attributes"]["label"] == "normal":
        normal += 1
    else:
        anomaly += 1
print(f"normal: {normal}, anomaly: {anomaly}")

{'AMOUNT_REQ': '20000', 'REG_DATE': '2011-10-01T00:38:44.546+02:00', 'concept:name': '173688', 'label': 'normal'}
normal: 9115, anomaly: 3972


In [34]:
import numpy as np

# Get the total number of cases
orig_num_cases = len(j["cases"])
print(f"Total number of orig cases: {orig_num_cases}")

# Sample 6000 cases or all cases if less than 6000
# sample_size = min(6000, num_cases)
# sampled_indices = np.random.choice(num_cases, sample_size, replace=False)

# Create j_sampled with the sampled cases
j_sampled = deepcopy(j)

# select only cases whole length is less than 15
j_sampled["cases"] = [j["cases"][i] for i in range(orig_num_cases) if len(j["cases"][i]["events"]) < 15]

new_num_cases = len(j_sampled["cases"])
print(f"Number of cases after filtering: {new_num_cases}")
sample_size = min(6000, new_num_cases)
sampled_indices = np.random.choice(new_num_cases, sample_size, replace=False)

j_sampled["cases"] = [j_sampled["cases"][i] for i in sampled_indices]

print(f"Number of final sampled cases: {len(j_sampled['cases'])}")

# Save the sampled data to a new file
with gzip.open(new_dataset_json_path, "w") as f:
    f.write(json.dumps(j_sampled).encode('utf-8'))

Total number of orig cases: 13087
Number of cases after filtering: 7032
Number of final sampled cases: 6000


In [35]:
def binet_to_df(path):
    with gzip.open(path, "r") as f:
        data = f.read()
        j = json.loads(data.decode('utf-8'))
    
    res_list = []
    
    for case in j['cases']:
        trace = pd.DataFrame.from_dict(case['events'])
        trace['anomaly'] = case['attributes']['label'] if isinstance(case['attributes']['label'], str) else case['attributes']['label']['anomaly']
        trace['trace_id'] = case['id']
        res_list.append(trace)
    
    if res_list:
        res = pd.concat(res_list, ignore_index=True)
        res = pd.concat([res.drop(['attributes'], axis=1), res['attributes'].apply(pd.Series)], axis=1)
    else:
        res = pd.DataFrame()
    
    return res

from datetime import datetime, timedelta

def assign_sequential_timestamps(df, start_time=None, step_minutes=10, duration_minutes=5):
    if start_time is None:
        start_time = datetime.now()

    df = df.copy()
    timestamps = []
    timestamps_end = []

    for trace_id, group in df.groupby('trace_id'):
        base_time = start_time
        for _ in range(len(group)):
            timestamps.append(base_time)
            timestamps_end.append(base_time + timedelta(minutes=duration_minutes))
            base_time += timedelta(minutes=step_minutes)

    df['timestamp'] = timestamps
    df['timestamp_end'] = timestamps_end
    return df

def convert_to_pm4py_df(df):
    df = df.copy()
    df = df.drop(["concept:name"], axis=1)
    df = df.rename(columns={'name': 'activity', 'trace_id':'case_id', 'user':'org:resource', 'anomaly': 'anomaly'})
    df = df.astype({'activity': str, 'anomaly': str, 'org:resource': str})
    df = format_dataframe(df, case_id='case_id',activity_key='activity', timestamp_key='timestamp')
    df = df.drop(['activity', 'timestamp', 'timestamp_end'], axis=1)
    return df

def convert_and_write_json_to_xes(path_to_json, path_to_xes):
    df = binet_to_df(path_to_json)
    df = assign_sequential_timestamps(df)
    df = convert_to_pm4py_df(df)
    write_xes(df, path_to_xes)

In [36]:
# dataset_names = ["medium", "small", "p2p", "paper"]
dataset_names = ["bpic12-0.3"]
dataset_json_path = r"D:\LTNcoder\.out\eventlogs\bpic12-0.3-1.json.gz"
dataset_xes_path = r"D:\LTNcoder\.out\eventlogs\bpic12-0.3-1.xes"
# convert_and_write_json_to_xes(dataset_json_path, dataset_xes_path)

In [37]:
mydf = binet_to_df(dataset_json_path)

In [38]:
mydf.shape

(33598, 9)

In [39]:
convertdf = convert_to_pm4py_df(mydf)

c:\Users\devas\anaconda3\envs\ltn\lib\site-packages\pm4py\utils.py:132: UserWarning: Some rows of the Pandas data frame have been removed because of empty case IDs, activity labels, or timestamps to ensure the correct functioning of PM4Py's algorithms.
  warnings.warn(
c:\Users\devas\anaconda3\envs\ltn\lib\site-packages\pm4py\utils.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[constants.CASE_CONCEPT_NAME] = df[constants.CASE_CONCEPT_NAME].astype(
c:\Users\devas\anaconda3\envs\ltn\lib\site-packages\pm4py\utils.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-do

In [40]:
convertdf.head(1)

,anomaly,case_id,lifecycle:transition,org:resource,case:concept:name,concept:name,time:timestamp,@@index,@@case_index
0,normal,173700,COMPLETE,112,173700,A_SUBMITTED+COMPLETE,2011-10-01 06:15:39.894000+00:00,0,0


In [41]:
write_xes(convertdf, dataset_xes_path)  

exporting log, completed traces ::   0%|          | 0/6000 [00:00<?, ?it/s]